In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_3.txt
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/movie_titles.csv
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_4.txt
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_1.txt
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/README
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/probe.txt
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_2.txt
/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/qualifying.txt


# Dataset Reading

In [2]:
from tqdm.auto import tqdm
import gc
import os

In [5]:
def load_netflix_file(file_path):
    rows = []

    current_movie = None

    with open(file_path, "r") as f:
        for line in tqdm(f, desc=os.path.basename(file_path)):
            line = line.strip()

            if line.endswith(":"):
                current_movie = int(line[:-1])

            else:
                user_id, rating, date = line.split(",")

                rows.append(
                    (
                        int(user_id),
                        current_movie,
                        int(rating),
                        date
                    )
                )

    df = pd.DataFrame(
        rows,
        columns=["UserID", "MovieID", "Rating", "Date"]
    )

    return df

In [6]:
paths = [
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_1.txt",
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_2.txt",
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_3.txt",
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/combined_data_4.txt"
]

dfs = []

for path in paths:
    df_temp = load_netflix_file(path)
    dfs.append(df_temp)

ratings_df = pd.concat(dfs, ignore_index=True)

del dfs
gc.collect()

combined_data_1.txt: 0it [00:00, ?it/s]

combined_data_2.txt: 0it [00:00, ?it/s]

combined_data_3.txt: 0it [00:00, ?it/s]

combined_data_4.txt: 0it [00:00, ?it/s]

92

In [7]:
ratings_df["UserID"] = ratings_df["UserID"].astype("int32")
ratings_df["MovieID"] = ratings_df["MovieID"].astype("int16")
ratings_df["Rating"] = ratings_df["Rating"].astype("int8")

ratings_df["Date"] = pd.to_datetime(
    ratings_df["Date"]
)

In [8]:
ratings_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100480507 entries, 0 to 100480506
Data columns (total 4 columns):
 #   Column   Dtype         
---  ------   -----         
 0   UserID   int32         
 1   MovieID  int16         
 2   Rating   int8          
 3   Date     datetime64[ns]
dtypes: datetime64[ns](1), int16(1), int32(1), int8(1)
memory usage: 1.4 GB


In [9]:
display(ratings_df.head())
ratings_df.shape

,UserID,MovieID,Rating,Date
0,1488844,1,3,2005-09-06
1,822109,1,5,2005-05-13
2,885013,1,4,2005-10-19
3,30878,1,4,2005-12-26
4,823519,1,3,2004-05-03


(100480507, 4)

In [10]:
ratings_df.to_parquet(
    "ratings_full.parquet",
    index=False
)

In [11]:
del ratings_df
gc.collect()

0

In [12]:
ratings_df = pd.read_parquet(
    "ratings_full.parquet"
)

In [17]:
print("Ratings :", len(ratings_df))
print("Users   :", ratings_df["UserID"].nunique())
print("Movies  :", ratings_df["MovieID"].nunique())

print()
print("Date Range")

print(ratings_df["Date"].min())
print(ratings_df["Date"].max())

Ratings : 100480507
Users   : 480189
Movies  : 17770

Date Range
1999-11-11 00:00:00
2005-12-31 00:00:00


In [19]:
user_counts = ratings_df.groupby(
    "UserID"
).size()

user_counts.describe(
    percentiles=[0.25,0.5,0.75,0.9,0.95,0.99]
)

count    480189.000000
mean        209.251997
std         302.339155
min           1.000000
25%          39.000000
50%          96.000000
75%         259.000000
90%         541.000000
95%         779.000000
99%        1390.000000
max       17653.000000
dtype: float64

## read movie_csv

In [3]:
movies = []

with open(
    "/kaggle/input/datasets/organizations/netflix-inc/netflix-prize-data/movie_titles.csv",
    encoding="latin-1"
) as f:

    for line in f:

        parts = line.strip().split(",")

        movie_id = int(parts[0])

        year = parts[1]

        title = ",".join(parts[2:])

        movies.append(
            [movie_id, year, title]
        )

movies = pd.DataFrame(
    movies,
    columns=[
        "MovieID",
        "Year",
        "Title"
    ]
)

movies["MovieID"] = pd.to_numeric(
    movies["MovieID"],
    errors="coerce"
)

movies["Year"] = pd.to_numeric(
    movies["Year"],
    errors="coerce"
)

movies.head()

,MovieID,Year,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


In [4]:
print(movies.shape)

print(
    movies["MovieID"].nunique()
)

(17770, 3)
17770


In [13]:
ratings_df["MovieID"].nunique()

movies["MovieID"].nunique()

set(
    ratings_df["MovieID"].unique()
) - set(
    movies["MovieID"].unique()
)

set()

In [14]:
movies.to_csv('movies.csv',index=False)